# Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window 

spark = (
    SparkSession.builder\
    .appName("spark-streaming")\
    .master("local[*]")\
    .config("spark.sql.shuffle.partitions", "4")\
    .getOrCreate()
)
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.sparkContext.setLogLevel("WARN")

print(f"spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/29 13:35:38 WARN Utils: Your hostname, youssef, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface enp1s0)
26/08/29 13:35:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/29 13:35:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark version: 4.1.3


# Build The Explicit Schema

In [2]:
event_schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("name", StringType(), nullable=True),
    StructField("category", StringType(), nullable=True),
    StructField("value", DoubleType(), nullable=True),
    StructField("timestamp", StringType(), nullable=True),
])
print("="*30)
print(f"event schema".center(30, "="))
print("="*30)
event_schema

=========event schema=========


StructType([StructField('id', IntegerType(), False), StructField('name', StringType(), True), StructField('category', StringType(), True), StructField('value', DoubleType(), True), StructField('timestamp', StringType(), True)])

# Read Stream

In [3]:
raw_stream_df = spark.readStream\
    .format("json")\
    .schema(event_schema)\
    .option("multiline", True)\
    .load("hdfs://localhost:9000/data/json/")

# print the schema of the streaming data
raw_stream_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: string (nullable = true)



# Bronze -> Silver Transformation 

In [4]:
silver_df = (
    raw_stream_df

    # -----------------------------
    # 1. Basic validation
    # -----------------------------
    .filter(
        col("name").isNotNull()
    )

    .withColumn(
        "is_valid_value",
        when(col("value") >= 0, lit(True))
        .otherwise(lit(False))
    )

    # -----------------------------
    # 2. Event time
    # -----------------------------
    .withColumn(
        "event_time",
        to_timestamp(
            col("timestamp"),
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )

    .filter(
        col("event_time").isNotNull()
    )

    # -----------------------------
    # 3. Watermark + deduplication
    # -----------------------------
    .withWatermark(
        "event_time",
        "10 minutes"
    )

    .dropDuplicatesWithinWatermark(
        ["id"]
    )

    # -----------------------------
    # 4. Date dimensions
    # -----------------------------
    .withColumn(
        "event_date",
        to_date(col("event_time"))
    )

    .withColumn(
        "event_year",
        year(col("event_time"))
    )

    .withColumn(
        "event_month",
        month(col("event_time"))
    )

    .withColumn(
        "event_quarter",
        quarter(col("event_time"))
    )

    .withColumn(
        "event_hour",
        hour(col("event_time"))
    )

    .withColumn(
        "event_day_name",
        date_format(col("event_time"), "EEEE")
    )

    .withColumn(
        "is_weekend",
        dayofweek(col("event_time")).isin(1, 7)
    )

    # -----------------------------
    # 5. Category normalization
    # -----------------------------
    .withColumn(
        "category",
        initcap(
            coalesce(
                trim(col("category")),
                lit("uncategorized")
            )
        )
    )

    # -----------------------------
    # 6. Event classification
    # -----------------------------
    .withColumn(
        "event_stage",
        when(
            col("name") == "order_placed",
            "1_ordered"
        )
        .when(
            col("name") == "payment_received",
            "2_paid"
        )
        .when(
            col("name") == "order_shipped",
            "3_shipped"
        )
        .when(
            col("name") == "item_returned",
            "4_returned"
        )
        .when(
            col("name") == "order_cancelled",
            "4_cancelled"
        )
        .otherwise("0_unknown")
    )

    .withColumn(
        "is_negative_event",
        col("name").isin(
            "item_returned",
            "order_cancelled"
        )
    )

    # -----------------------------
    # 7. Value transformations
    # -----------------------------
    .withColumn(
        "value",
        round(col("value"), 2)
    )

    .withColumn(
        "value_tier",
        when(
            col("value") < 0,
            "invalid"
        )
        .when(
            col("value") < 100,
            "low"
        )
        .when(
            col("value") < 300,
            "medium"
        )
        .otherwise("high")
    )

    .withColumn(
        "is_high_value",
        col("value_tier") == "high"
    )

    # -----------------------------
    # 8. Metadata
    # -----------------------------
    .withColumn(
        "ingested_at",
        current_timestamp()
    )

    .withColumn(
        "source_system",
        lit("hdfs_json")
    )

    # -----------------------------
    # 9. Final Silver schema
    # -----------------------------
    .select(
        "id",
        "name",
        "event_stage",
        "is_negative_event",
        "category",
        "value",
        "value_tier",
        "is_high_value",
        "is_valid_value",
        "event_time",
        "event_date",
        "event_year",
        "event_month",
        "event_quarter",
        "event_hour",
        "event_day_name",
        "is_weekend",
        "ingested_at",
        "source_system"
    )
)

# Gold Layer

In [5]:
gold_df = (
    silver_df
    .groupBy(
        window(
            col("event_time"),
            "5 minutes"
        ),
        col("category")
    )
    .agg(
        count("*").alias("event_count"),

        round(
            sum("value"),
            2
        ).alias("total_value"),

        round(
            avg("value"),
            2
        ).alias("avg_value")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("category"),
        col("event_count"),
        col("total_value"),
        col("avg_value")
    )
)

In [6]:
gold_returns_df = (
    silver_df
    .withWatermark("event_time", "10 minutes")

    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("category")
    )

    .agg(
        count("*").alias("total_events"),

        sum(
            col("is_negative_event").cast("int")
        ).alias("negative_events")
    )

    .withColumn(
        "negative_event_rate_pct",
        round(
            (
                col("negative_events") /
                col("total_events")
            ) * 100,
            2
        )
    )

    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("category"),
        col("total_events"),
        col("negative_events"),
        col("negative_event_rate_pct")
    )
)

# Sinks

In [7]:
PARQUET_OUTPUT_PATH = "hdfs://localhost:9000/data/output/parquet/silver_events"
PARQUET_CHECKPOINT_PATH = "hdfs://localhost:9000/data/checkpoints/parquet/silver_events"

JSON_OUTPUT_PATH = "hdfs://localhost:9000/data/output/json/silver_events"
JSON_CHECKPOINT_PATH = "hdfs://localhost:9000/data/checkpoints/json/silver_events"

GOLD_JSON_OUTPUT_PATH = "hdfs://localhost:9000/data/output/json/gold_category_windows"
GOLD_JSON_CHECKPOINT_PATH = "hdfs://localhost:9000/data/checkpoints/json/gold_category_windows"

GOLD_RETURNS_OUTPUT_PATH = "hdfs://localhost:9000/data/output/json/gold_negative_event_rate"
GOLD_RETURNS_CHECKPOINT_PATH = "hdfs://localhost:9000/data/checkpoints/json/gold_negative_event_rate"

In [8]:
parquet_query = (
    silver_df.writeStream
    .format("parquet")
    .option("path", PARQUET_OUTPUT_PATH)
    .option("checkpointLocation", PARQUET_CHECKPOINT_PATH)
    .outputMode("append")
    .partitionBy("event_date", "category")
    .trigger(processingTime="10 seconds")
    .queryName("silver_events_to_parquet")
    .start()
)

In [9]:
json_query = (
    silver_df.writeStream
    .format("json")
    .option("path", JSON_OUTPUT_PATH)
    .option("checkpointLocation", JSON_CHECKPOINT_PATH)
    .outputMode("append")
    .partitionBy("event_date")
    .trigger(processingTime="10 seconds")
    .queryName("silver_events_to_json")
    .start()
)

In [10]:
gold_query = (
    gold_df.writeStream
    .format("json")
    .option("path", GOLD_JSON_OUTPUT_PATH)
    .option("checkpointLocation", GOLD_JSON_CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(processingTime="30 seconds")
    .queryName("gold_category_windows_to_json")
    .start()
)

26/08/29 13:35:47 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/08/29 13:35:47 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


In [11]:
gold_returns_query = (
    gold_returns_df.writeStream
    .format("json")
    .option("path", GOLD_RETURNS_OUTPUT_PATH)
    .option("checkpointLocation", GOLD_RETURNS_CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(processingTime="30 seconds")
    .queryName("gold_negative_event_rate_to_json")
    .start()
)

26/08/29 13:35:47 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


In [ ]:
parquet_query.awaitTermination()
json_query.awaitTermination()
gold_query.awaitTermination()
gold_returns_query.awaitTermination()

26/08/29 13:35:47 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/08/29 13:36:00 ERROR MicroBatchExecution: Query gold_negative_event_rate_to_json [id = 26aded94-3a1b-4024-8c3d-74b98a0dba66, runId = 1b259360-8526-4f88-8b6d-dd9baddc7b5a] terminated with error
org.apache.spark.sql.AnalysisException: Redefining watermark is disallowed. You can set the config 'spark.sql.streaming.statefulOperator.allowMultiple' to 'false' to restore the previous behavior. Note that multiple stateful operators will be disallowed.
	at org.apache.spark.sql.execution.streaming.runtime.PropagateWatermarkSimulator.$anonfun$doSimulate$1(WatermarkPropagator.scala:212)
	at org.apache.spark.sql.execution.streaming.runtime.PropagateWatermarkSimulator.$anonfun$doSimulate$1$adapted(WatermarkPropagator.scala:203)
	at org.apache.spark.sql.catalyst.trees.TreeNode.foreachUp(TreeNode.scala:273)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1(TreeNode.scala:2